#### Libraries

In [ ]:
import pandas as pd
import os
from tqdm import tqdm
import matplotlib.pyplot as plt
import plotly.express as px
import requests
import time
import xml.etree.ElementTree as ET
import numpy as np
from sklearn.preprocessing import MinMaxScaler

## Data

#### PPI

In [ ]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [ ]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

In [ ]:
protein_interaction

#### Drug Bank

In [ ]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [ ]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [ ]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

In [ ]:
print(f"{len(set(Drug_bank['drug']))} unique drugs found in DrugBank.")
print(f"{len(set(Drug_bank['gene']))} unique genes found in DrugBank.")
print(f"{len(set(tuple(Drug_bank['drug'] + Drug_bank['gene'])))} unique drug-gene pairs found in DrugBank.")
print(f"{len(Drug_bank)} size of DrugBank interactions found in DrugBank.")

#### CNV Results

In [ ]:
hpv_positive_amplification_gene_df = pd.read_csv('Results/CNV results/HPV positive amplification genes.csv')
hpv_positive_top_amplification_genes = list(hpv_positive_amplification_gene_df['gene_name'].values)

hpv_positive_deletion_gene_df = pd.read_csv('Results/CNV results/HPV positive deletion genes.csv')
hpv_positive_top_deletion_genes = list(hpv_positive_deletion_gene_df['gene_name'].values)

hpv_negative_amplification_gene_df = pd.read_csv('Results/CNV results/HPV negative amplification genes.csv')
hpv_negative_top_amplification_genes = list(hpv_negative_amplification_gene_df['gene_name'].values)

hpv_negative_deletion_gene_df = pd.read_csv('Results/CNV results/HPV negative deletion genes.csv')
hpv_negative_top_deletion_genes = list(hpv_negative_deletion_gene_df['gene_name'].values)

In [ ]:
hpv_positive_amplification_gene_df

## Identify top genes

In [ ]:
def plot_score_distribution(df):
    """
    Function to plot the distribution of scores in a DataFrame.
    To allow for visual determination of the best cutoff for each score.
    The scores analyzed are GISTIC2.0, frequency_percentage, gistic_score, p-value, and q-value.
    """

    # Get the df variable name to get the title of the plot
    title = f"Distribution of Scores with mean and 95th Percentile"

    scores = ['frequency_percentage', 'gistic_score', 'q_value', 'empirical_q_value']
    #df = df[(df['frequency_percentage'] != 0) & (df['gistic_score'] != 0)]
    ### want larger plots to see all 100 bins
    plt.figure(figsize=(15, 10))
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.subplots_adjust(hspace=0.4, wspace=0.4)
    for i, score in enumerate(scores):
        plt.subplot(3, 2, i + 1)
        ### plot the histogram with 100 bins
        ### set alpha to 0.7 for better visibility
        ### set color to blue and edgecolor to black for better visibility
        ### set yscale to log for better visibility of the distribution
        plt.hist(df[score], bins=100, alpha=0.7, color='blue', edgecolor='black')
        if score == 'empirical_q_value' or score == 'q_value':
            plt.axvline(.05, color='red', linestyle='dashed', linewidth=1, label = 'Significance Threshold: 0.05')
            #plt.axvline(df[score].quantile(0.05), color='green', linestyle='dashed', linewidth=1, label = f'95th Percentile of {score}: {df[score].quantile(0.05):.4f}')
        else:
            plt.axvline(df[score].mean(), color='red', linestyle='dashed', linewidth=1, label = f'Mean of {score}: {df[score].mean():.4f}')
            plt.axvline(df[score].quantile(0.95), color='green', linestyle='dashed', linewidth=1, label = f'95th Percentile of {score}: {df[score].quantile(0.95):.4f}')
        plt.title(f'Distribution of {score}')
        plt.xlabel(score)
        plt.ylabel('Frequency')
        plt.legend()
        plt.yscale('log')
        plt.grid(axis='y', alpha=0.75)    
    plt.show()

In [ ]:
def plot_score_distribution_with_cutoff(df, cutoff):
    """
    Function to plot the distribution of scores in a DataFrame with a cutoff line.
    The scores analyzed are GISTIC2.0, frequency_percentage, gistic_score, p-value, and q-value.
    These scores are a dictionary with the cutoff values for each score. keys are the score names and values are the cutoff values.
    """
    # Get the df variable name to get the title of the plot
    # df_name = [name for name in globals() if globals()[name] is df][0]
    # print(df_name)
    title = f"Distribution of Scores  with Cutoffs"
    # Remove any existing figure to avoid overlapping titles in Jupyter
    plt.close('all')
    scores = ['frequency_percentage', 'gistic_score', 'q_value', 'empirical_q_value']
    plt.figure(figsize=(15, 10))
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.subplots_adjust(top=0.9)  # Adjust the top to make space for the title
    for i, score in enumerate(scores):
        if score not in cutoff:
            continue
        plt.subplot(3, 2, i + 1)
        plt.hist(df[score], bins=100, alpha=0.7, color='blue', edgecolor='black')
        if score == 'empirical_q_value' or score == 'q_value':
            plt.axvline(.05, color='red', linestyle='dashed', linewidth=1, label = 'Significance Threshold: 0.05')
            #plt.axvline(cutoff[score], color='red', linestyle='--', label=f'Cutoff: {cutoff[score]}')
            plt.axvline(df[score].quantile(0.05), color='green', linestyle='dashed', linewidth=1, label = f'95th Percentile of {score}: {df[score].quantile(0.05):.4f}')
        elif df[score].mean() < 0:
            plt.axvline(x=cutoff[score], color='red', linestyle='--', label=f'Cutoff: {cutoff[score]}')
            plt.axvline(df[score].quantile(.95), color='green', linestyle='dashed', linewidth=1, label=f'5th Percentile of {score}: {df[score].quantile(.95):.4f}')
        else:
            plt.axvline(x=cutoff[score], color='red', linestyle='--', label=f'Cutoff: {cutoff[score]}')
            plt.axvline(df[score].quantile(.95), color='green', linestyle='dashed', linewidth=1, label=f'95th Percentile of {score}: {df[score].quantile(.95):.4f}')
        plt.title(f'Distribution of {score}')
        plt.xlabel(score)
        plt.ylabel('Frequency')
        plt.legend()
        plt.yscale('log')
        plt.grid(axis='y', alpha=0.75)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

### HPV+

#### Amp

In [ ]:
hpv_positive_amplification_gene_df

In [ ]:
### plot distribution of gistic scores vs frequency percatage
plt.figure(figsize=(10, 6))
plt.scatter(hpv_positive_amplification_gene_df['gistic_score'], hpv_positive_amplification_gene_df['frequency_percentage'], alpha=0.5, color='blue')
plt.title('GISTIC Scores vs Frequency Percentage for HPV Positive Amplification Genes')
plt.xlabel('GISTIC Score')
plt.ylabel('Frequency Percentage')
plt.axhline(y=hpv_positive_amplification_gene_df['frequency_percentage'].mean(), color='red', linestyle='dashed', linewidth=1, label=f'Mean Frequency Percentage: {hpv_positive_amplification_gene_df["frequency_percentage"].mean():.2f}')
plt.legend()
plt.show()

In [ ]:
## plot gistic score vs q value
plt.figure(figsize=(10, 6))
plt.scatter(hpv_positive_amplification_gene_df['gistic_score'], hpv_positive_amplification_gene_df['q_value'], alpha=0.5, color='blue')
plt.title('GISTIC Scores vs q-value for HPV Positive Amplification Genes')
plt.xlabel('GISTIC Score')
plt.ylabel('q-value')
plt.axhline(y=0.05, color='red', linestyle='dashed', linewidth=1, label='Significance Threshold: 0.05')
plt.legend()
plt.show()

In [ ]:
hpv_positive_amplification_gene_df

In [ ]:
## using plotly plot the distribution of gistic SCORES by gene name
def plot_gistic_distribution(df, title):
    ### plotly will add dots for significant incr
    import plotly.express as px
    fig = px.box(df, x='gene_name', y='gistic_score', title=title)
    ### add horizontal line cut off at .01
    fig.add_hline(y=0.29, line_dash="dash", line_color="red", annotation_text="Cutoff: 0.01", annotation_position="top left")
    fig.show()

In [ ]:
plot_gistic_distribution(hpv_positive_amplification_gene_df, "HPV Positive Amplification Genes GISTIC2.0 Score Distribution")

In [ ]:
### plot Amplification_sum (i.e. total log2(amplification CNV) across samples) of hpv_positive_amplification_gene_df
def plot_amplification_sum(df, title):
    """
    Function to plot the amplification sum of a DataFrame.
    The amplification sum is the total log2(amplification CNV) across samples.
    """
    plt.figure(figsize=(10, 6))
    plt.bar(df['gene_name'], df['Amplification_sum (i.e. total log2(amplification CNV) across samples)'], color='blue', edgecolor='black')
    plt.axhline(y=0.01, color='red', linestyle='dashed', linewidth=1, label='Cutoff: 0.01')
    plt.title(title)
    plt.xlabel('Gene Name')
    plt.ylabel('Amplification Sum')
    plt.xticks(rotation=90)
    plt.legend()
    plt.grid(axis='y', alpha=0.75)
    plt.show()

In [ ]:
### plot histogram of Amplification_sum (i.e. total log2(amplification CNV) across samples) in hpv_positive_amplification_gene_df
plt.Figure(figsize=(10, 6))
plt.hist(hpv_positive_amplification_gene_df['Amplification_sum (i.e. total log2(amplification CNV) across samples)'], bins=100, alpha=0.7, color='blue', edgecolor='black')
#plt.axvline(0.01, color='red', linestyle='dashed', linewidth=1, label='Cutoff: 0.01')
plt.title('Distribution of Amplification Sum')
plt.xlabel('Amplification Sum')
plt.ylabel('Frequency')
plt.legend()
plt.grid(axis='y', alpha=0.75)
plt.show()

In [ ]:
plot_score_distribution(hpv_positive_amplification_gene_df)

In [ ]:
### decided cut offs
cutoffs = {
    'frequency_percentage': 30,
    'gistic_score': 0.296,
    'empirical_q_value': 0.05,
    'q_value': .05
}


In [ ]:
plot_score_distribution_with_cutoff(hpv_positive_amplification_gene_df, cutoffs)

In [ ]:
hpv_positive_amplification_top_gene_df = hpv_positive_amplification_gene_df[
    (hpv_positive_amplification_gene_df['frequency_percentage'] >= cutoffs['frequency_percentage']) &
    (hpv_positive_amplification_gene_df['gistic_score'] >= cutoffs['gistic_score']) &
    (hpv_positive_amplification_gene_df['q_value'] <= cutoffs['q_value']) &
    (hpv_positive_amplification_gene_df['empirical_q_value'] <= cutoffs['empirical_q_value'])
].copy()

In [ ]:
hpv_positive_amplification_top_gene_df

In [ ]:
hpv_positive_amplification_top_gene_df.to_csv('Results/CNV results/HPV positive amplification top genes.csv', index=False)

#### Del

In [ ]:
hpv_positive_deletion_gene_df

In [ ]:
plot_score_distribution(hpv_positive_deletion_gene_df)

In [ ]:
cutoffs = {
    'frequency_percentage': 12,
    'gistic_score': .091,
    'empirical_q_value': 0.05,
    'q_value': 0.05
}

In [ ]:
plot_score_distribution_with_cutoff(hpv_positive_deletion_gene_df, cutoffs)

In [ ]:
### plot the gistic score distribution
plot_gistic_distribution(hpv_positive_deletion_gene_df, "GISTIC Score Distribution for HPV Positive Deletions")

In [ ]:
hpv_positive_deletion_top_gene_df = hpv_positive_deletion_gene_df[
    (hpv_positive_deletion_gene_df['frequency_percentage'] >= cutoffs['frequency_percentage']) &
    (hpv_positive_deletion_gene_df['gistic_score'] >= cutoffs['gistic_score']) &
    (hpv_positive_deletion_gene_df['q_value'] <= cutoffs['q_value']) &
    (hpv_positive_deletion_gene_df['empirical_q_value'] <= cutoffs['empirical_q_value'])
].copy()

In [ ]:
### plot gene vs gistic score
hpv_positive_deletion_top_gene_df.sort_values(by='gistic_score', ascending=False, inplace=True)
### make the plot larger to see all the genes
plt.figure(figsize=(25, 25))
plt.barh(hpv_positive_deletion_top_gene_df['gene_name'], hpv_positive_deletion_top_gene_df['gistic_score'], color='blue', edgecolor='black')
plt.axvline(x=cutoffs['gistic_score'], color='red', linestyle='--', label=f'Cutoff: {cutoffs["gistic_score"]}')
plt.title('Top HPV Positive Deletion Genes by GISTIC Score')
plt.xlabel('GISTIC Score')
plt.ylabel('Gene Name')
plt.legend()
plt.show()

In [ ]:
hpv_positive_deletion_top_gene_df

In [ ]:
len(set(hpv_positive_deletion_top_gene_df['gene_name']))

In [ ]:
hpv_positive_deletion_top_gene_df.to_csv('Results/CNV results/HPV positive deletion top genes.csv', index=False)

### HPV-

#### Amp

In [ ]:
hpv_negative_amplification_gene_df

In [ ]:
plot_score_distribution(hpv_negative_amplification_gene_df)

In [ ]:
cutoffs = {
    'frequency_percentage': 34.5,
    'gistic_score': 0.3,
    'empirical_q_value': 0.05,
    'q_value': 0.05
}

In [ ]:
plot_score_distribution_with_cutoff(hpv_negative_amplification_gene_df, cutoffs)

In [ ]:
hpv_negative_amplification_top_gene_df = hpv_negative_amplification_gene_df[
    (hpv_negative_amplification_gene_df['frequency_percentage'] >= cutoffs['frequency_percentage']) &
    (hpv_negative_amplification_gene_df['gistic_score'] >= cutoffs['gistic_score']) &
    (hpv_negative_amplification_gene_df['q_value'] <= cutoffs['q_value']) &
    (hpv_negative_amplification_gene_df['empirical_q_value'] <= cutoffs['empirical_q_value'])
].copy()

In [ ]:
hpv_negative_amplification_top_gene_df

In [ ]:
hpv_negative_amplification_top_gene_df.to_csv('Results/CNV results/HPV negative amplification top genes.csv', index=False)

In [ ]:
### plot gene vs gistic score
plt.figure(figsize=(25, 25))
plt.barh(hpv_negative_amplification_top_gene_df['gene_name'], hpv_negative_amplification_top_gene_df['gistic_score'], color='blue', edgecolor='black')
plt.axvline(x=cutoffs['gistic_score'], color='red', linestyle='--', label=f'Cutoff: {cutoffs["gistic_score"]}')
plt.title('Top HPV Negative Amplification Genes by GISTIC Score')
plt.xlabel('GISTIC Score')
plt.ylabel('Gene Name')
plt.legend()
plt.show()

In [ ]:
hpv_neg_amp_genes = hpv_negative_amplification_top_gene_df['gene_name'].unique()
print(f"Number of unique genes in HPV negative amplification top genes: {len(hpv_neg_amp_genes)}")
drugbank_genes = list(set(Drug_bank['gene'].dropna().values))
overlap_genes = set(hpv_neg_amp_genes) & set(drugbank_genes)
print(f"Number of overlapping genes between HPV negative amplification top genes and DrugBank genes: {len(overlap_genes)}")
print(f"Overlapping genes: {overlap_genes}")

## immediate neighbors of these genes in the PPI network
hpv_neg_amp_genes = hpv_negative_amplification_top_gene_df['gene_name'].unique()
ppi_genes = list(set(protein_interaction['Translated_protein_1'].dropna().values) | set(protein_interaction['Translated_protein_2'].dropna().values))
overlap_genes = set(hpv_neg_amp_genes) & set(ppi_genes)
#print(f"Number of overlapping genes between HPV negative amplification top genes and PPI genes: {len(overlap_genes)}")
# print(f"Overlapping genes: {overlap_genes}")

## number of immediate neighbors of the top genes in the PPI network
usable_protein_interaction = protein_interaction[protein_interaction['combined_score'] > 700]
protein_interaction_neighbors = usable_protein_interaction[usable_protein_interaction['Translated_protein_1'].isin(overlap_genes)]
### remove duplicates and hpv neg amp genes from the neighbors
immediate_neighbors = protein_interaction_neighbors['Translated_protein_2'].unique()
print(f"Number of immediate neighbors of the top genes in the PPI network: {len(immediate_neighbors)}")

### number of immediate neighbors or key risk genes that are in drugbank
# This will give us a list of all the immediate neighbors and key risk genes that we want to check for overlap with drugbank
immediate_neighbors_and_risk = list(list(immediate_neighbors) + list(hpv_neg_amp_genes)) 
### this will give us the number of immediate neighbors or key risk genes that are in drugbank
immediate_neighbors_and_risk_in_drugbank = set(immediate_neighbors_and_risk) & set(drugbank_genes)
print(f"Number of immediate neighbors or key risk genes that are in DrugBank: {len(set(immediate_neighbors_and_risk_in_drugbank))}")
#print(f"Immediate neighbors or key risk genes that are in DrugBank: {immediate_neighbors_and_risk_in_drugbank}")

#### Del

In [ ]:
hpv_negative_deletion_gene_df

In [ ]:
plot_score_distribution(hpv_negative_deletion_gene_df)

In [ ]:
cutoffs = {
    'frequency_percentage': 19,
    'gistic_score': .0845,
    'empirical_q_value': 0.05,
    'q_value': 0.05
}

In [ ]:
plot_score_distribution_with_cutoff(hpv_negative_deletion_gene_df, cutoffs)

In [ ]:
hpv_negative_deletion_top_gene_df = hpv_negative_deletion_gene_df[
    (hpv_negative_deletion_gene_df['frequency_percentage'] >= cutoffs['frequency_percentage']) &
    (hpv_negative_deletion_gene_df['gistic_score'] >= cutoffs['gistic_score']) &
    (hpv_negative_deletion_gene_df['q_value'] <= cutoffs['q_value']) &
    (hpv_negative_deletion_gene_df['empirical_q_value'] <= cutoffs['empirical_q_value'])
].copy()

In [ ]:
hpv_negative_deletion_top_gene_df

In [ ]:
### plot gene vs gistic scores
plt.figure(figsize=(25, 25))
plt.barh(hpv_negative_deletion_top_gene_df['gene_name'], hpv_negative_deletion_top_gene_df['gistic_score'], color='blue', edgecolor='black')
plt.axvline(x=cutoffs['gistic_score'], color='red', linestyle='--', label=f'Cutoff: {cutoffs["gistic_score"]}')
plt.title('Top HPV Negative Deletion Genes by GISTIC Score')
plt.xlabel('GISTIC Score')
plt.ylabel('Gene Name')
plt.legend()
plt.show()

In [ ]:
hpv_neg_del_genes = hpv_negative_deletion_top_gene_df['gene_name'].unique()
print(f"Number of unique genes in HPV negative deletion top genes: {len(hpv_neg_del_genes)}")
drugbank_genes = list(set(Drug_bank['gene'].dropna().values))
overlap_genes = set(hpv_neg_del_genes) & set(drugbank_genes)
print(f"Number of overlapping genes between HPV negative deletion top genes and DrugBank genes: {len(overlap_genes)}")
print(f"Overlapping genes: {overlap_genes}")

## immediate neighbors of these genes in the PPI network
hpv_neg_del_genes = hpv_negative_deletion_top_gene_df['gene_name'].unique()
ppi_genes = list(set(protein_interaction['Translated_protein_1'].dropna().values) | set(protein_interaction['Translated_protein_2'].dropna().values))
overlap_genes = set(hpv_neg_del_genes) & set(ppi_genes)
#print(f"Number of overlapping genes between HPV negative deletion top genes and PPI genes: {len(overlap_genes)}")
# print(f"Overlapping genes: {overlap_genes}")

## number of immediate neighbors of the top genes in the PPI network
usable_protein_interaction = protein_interaction[protein_interaction['combined_score'] > 700]
protein_interaction_neighbors = usable_protein_interaction[usable_protein_interaction['Translated_protein_1'].isin(overlap_genes)]
### remove duplicates and hpv neg del genes from the neighbors
immediate_neighbors = protein_interaction_neighbors['Translated_protein_2'].unique()
print(f"Number of immediate neighbors of the top genes in the PPI network: {len(immediate_neighbors)}")

### number of immediate neighbors or key risk genes that are in drugbank
# This will give us a list of all the immediate neighbors and key risk genes that we want to check for overlap with drugbank
immediate_neighbors_and_risk = list(list(immediate_neighbors) + list(hpv_neg_del_genes)) 
immediate_neighbors_and_risk_in_drugbank = set(immediate_neighbors_and_risk) & set(drugbank_genes)
print(f"Number of immediate neighbors or key risk genes that are in DrugBank: {len(immediate_neighbors_and_risk_in_drugbank)}")
print(f"Immediate neighbors or key risk genes that are in DrugBank: {immediate_neighbors_and_risk_in_drugbank}")

In [ ]:
hpv_negative_deletion_top_gene_df.to_csv('Results/CNV results/HPV negative deletion top genes.csv', index=False)

In [ ]:
top_hpv_negative_deletion_genes = list(hpv_negative_deletion_top_gene_df['gene_name'].values)

In [ ]:
# Drug_bank[Drug_bank['gene'].isin(top_hpv_negative_deletion_genes)]